# 42 — Power tools

Three new tools from this release — each pairs naturally with one or more of the specialists from notebook 41.

| Tool | Who it's for | Auth |
|---|---|---|
| `computer_use` | Anyone driving a native GUI (macOS / Linux / Windows) | None — uses platform primitives |
| `hubspot_ops` | Sales, customer success | `HUBSPOT_TOKEN` env |
| `research_brief` | Researcher, PM, marketing, sales | None — public web search |


In [ ]:
from pathlib import Path
import sys

# Resolve the repo root whether this cell is running from ./notebooks
# or from the repo root — mirrors the existing 01–36 notebook series.
ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_llm_from_env

# Bedrock Llama 4 Scout is the default model this library ships with
# (see `shipit_agent/config.py`). No model name required — the helper
# reads `AWS_REGION` / credentials from env and returns an adapter that
# works with every Autopilot / Agent class.
llm = build_llm_from_env("bedrock")
print("Bedrock LLM ready:", type(llm).__name__)


## 1. `computer_use` — desktop automation

`computer_use` is the library's Claude-Desktop-style primitive. It drives the LOCAL desktop via platform-native helpers:

- macOS → `screencapture` + `cliclick` (or AppleScript fallback)
- Linux → `scrot` / ImageMagick `import` + `xdotool`
- Windows → PowerShell

In [ ]:
from shipit_agent.tools.computer_use import ComputerUseTool

computer = ComputerUseTool()
print("Schema enum:", computer.schema()['function']['parameters']['properties']['action']['enum'])
print("Output dir:", computer.output_dir)


### 1a. Take a screenshot

When run on your laptop, this actually captures the screen. In this notebook it'll succeed only if `screencapture` (macOS) / `scrot` (linux) / powershell (windows) is on PATH — otherwise you'll get a clear install hint.

In [ ]:
from shipit_agent.tools.base import ToolContext

out = computer.run(ToolContext(prompt="demo"), action="screenshot")
print(out.text)


### 1b. Use it inside an Autopilot

For a tutorial / QA run, combine `computer_use` with a specialist and Autopilot: the agent drives the GUI, the runtime keeps it on track with a budget.

In [ ]:
from shipit_agent import Agent
from shipit_agent.agents import AgentRegistry

registry = AgentRegistry()
def_ = registry.get("design-reviewer")
ui_agent = Agent(
    llm=llm, prompt=def_.prompt, tools=[computer],
    max_iterations=6, name="UI Reviewer",
)
# In your own environment, replace the prompt with something that
# drives your app (e.g. open Preferences, verify toggles, screenshot).
print("UI Reviewer agent ready:", ui_agent.name)


## 2. `research_brief` — one-call research primitive

Search the web, skim the top N pages, return a structured brief with numbered citations. No API key — uses DuckDuckGo HTML.

In [ ]:
from shipit_agent.tools.research_brief import ResearchBriefTool

research = ResearchBriefTool()
out = research.run(ToolContext(prompt="demo"), query="uv vs poetry python package manager 2026", max_sources=4)
print(out.text)


Pass `deep=True` to also fetch each result page and embed a short summary. Use sparingly — one call fans out to N+1 HTTP requests.

In [ ]:
out = research.run(ToolContext(prompt="demo"), query="SQLite vs DuckDB for analytics", max_sources=3, deep=True)
print(out.text[:1200])


## 3. `hubspot_ops` — CRM in one tool

One tool, many actions: `search_contacts`, `search_companies`, `search_deals`, `get_*`, `create_contact`, `create_deal`, `add_note`, `list_owners`. Auth is `HUBSPOT_TOKEN` env (private-app bearer token).

In [ ]:
import os
from shipit_agent.tools.hubspot import HubspotTool

if os.environ.get("HUBSPOT_TOKEN"):
    hubspot = HubspotTool()
    out = hubspot.run(ToolContext(prompt="demo"), action="search_contacts", query="test", limit=5)
    print(out.text[:800])
else:
    print("Set HUBSPOT_TOKEN to hit live HubSpot — schema still works for dry-run inspection:")
    from shipit_agent.tools.hubspot import HubspotTool as H
    print(H(token=None).schema()['function']['parameters']['properties']['action']['enum'])


## 4. Stacking tools in one long-running Autopilot

A single agent with all three tools plus a substantial budget can drive a surprisingly complex outbound-sales flow end to end: research accounts (research_brief), log activity (hubspot_ops), take screenshots for internal review (computer_use). Budgets make it safe to leave running.

In [ ]:
from shipit_agent.autopilot import Autopilot, BudgetPolicy
from shipit_agent.deep import Goal

tools = [research, computer]   # add HubspotTool() when token is set
autopilot = Autopilot(
    llm=llm,
    goal=Goal(
        objective="Compare three vendors of time-series databases and produce a recommendation.",
        success_criteria=[
            "Each vendor has a 1-paragraph summary with citations",
            "A comparison table covering licensing + scale + pricing",
            "A 1-paragraph recommendation with the primary reason",
        ],
    ),
    tools=tools,
    budget=BudgetPolicy(max_iterations=8, max_seconds=600, max_tool_calls=30, max_dollars=3.0),
)
print("Ready to run. autopilot.run(run_id='tsdb-shootout') when you're ready.")


## Summary

Each tool is designed to be usable standalone, but the real leverage comes from handing them to an Autopilot with a thoughtful budget. That's the Claude-Desktop model: long-running, goal-directed, live-streamed, budget-gated.

Full series complete. Thanks for reading — now go build.